In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np

In [0]:
# Load gold table
df_gold = spark.table("workspace.taxi_schema.taxi_gold")

print(f"Gold table records: {df_gold.count():,}")
print(f"\nSchema:")
df_gold.printSchema()
print(f"\nSample data:")
display(df_gold.limit(10))

In [0]:
# Convert to pandas for sklearn (data is small enough - 3,286 records)
df_pandas = df_gold.toPandas()

print(f"Dataset shape: {df_pandas.shape}")

# Prepare features (X) and target (y)
X = df_pandas[['pickup_hour', 'PULocationID']]
X = pd.get_dummies(X, columns=['PULocationID'], prefix='PULocationID')
y = df_pandas['avg_fare_per_mile']
print(f"after dummy X shape: {X.shape}")

# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set: {len(X_train):,} records")
print(f"Test set: {len(X_test):,} records")

# Train linear regression model
print(f"\nTraining linear regression model...")
model = LinearRegression()
model.fit(X_train, y_train)

print("✓ Model trained successfully")
print(f"\nModel coefficients:")
print(f"  Number of encoded features: {X_train.shape[1]}")
print(f"  Hour coefficient: {model.coef_[0]:.6f}")
print(f"  Intercept: {model.intercept_:.6f}")

In [0]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"R-squared (R²): {r2:.4f}")
print(f"  → Explains {r2*100:.2f}% of variance in fare per mile")
print(f"\nRoot Mean Squared Error (RMSE): ${rmse:.4f}")
print(f"Mean Absolute Error (MAE): ${mae:.4f}")
print("=" * 60)

In [0]:
# Retrain model on ENTIRE dataset for best predictions
print("Retraining model on full dataset...")
X_full = df_pandas[['pickup_hour', 'PULocationID']]
X_full = pd.get_dummies(X_full, columns=['PULocationID'], prefix='PULocationID')
y_full = df_pandas['avg_fare_per_mile']

model_full = LinearRegression()
model_full.fit(X_full, y_full)
print("✓ Model retrained on full dataset")

# Generate predictions for all records using full model
y_all_pred = model_full.predict(X_full)

# Create analysis result dataframe
df_analysis_pandas = df_pandas.copy()
df_analysis_pandas['predicted_fare_per_mile'] = y_all_pred
df_analysis_pandas['actual_fare_per_mile'] = df_pandas['avg_fare_per_mile']
df_analysis_pandas['prediction_error'] = df_analysis_pandas['predicted_fare_per_mile'] - df_analysis_pandas['actual_fare_per_mile']
df_analysis_pandas['absolute_error'] = np.abs(df_analysis_pandas['prediction_error'])

# Select and reorder columns for dashboard
df_result = df_analysis_pandas[[
    'PULocationID',
    'pickup_hour',
    'predicted_fare_per_mile',
    'actual_fare_per_mile',
    'prediction_error',
    'absolute_error',
    'trip_count',
    'avg_trip_duration',
    'avg_trip_distance'
]]

# Convert back to Spark DataFrame and save to Delta
df_analysis_spark = spark.createDataFrame(df_result)
df_analysis_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_schema.analysis_result")

print(f"\n✓ Created analysis_result table with {len(df_result):,} records")
print(f"✓ Table ready for dashboard visualization!")
print(f"\nSample results:")
print(df_result.head(20).to_string())

In [0]:
# Overall prediction quality
print("=" * 60)
print("PREDICTION QUALITY SUMMARY")
print("=" * 60)
print(f"Average Absolute Error: ${df_result['absolute_error'].mean():.4f}")
print(f"Max Absolute Error: ${df_result['absolute_error'].max():.4f}")
print(f"Min Absolute Error: ${df_result['absolute_error'].min():.4f}")
print(f"Std Dev of Error: ${df_result['absolute_error'].std():.4f}")
print("=" * 60)

# Top 10 location-hour combinations with highest prediction errors
print("\nTop 10 Location-Hour Combinations with Highest Prediction Errors:")
worst_predictions = df_result.nlargest(10, 'absolute_error')[[
    'PULocationID', 'pickup_hour', 'actual_fare_per_mile',
    'predicted_fare_per_mile', 'absolute_error', 'trip_count'
]]
print(worst_predictions.to_string())

# Prediction accuracy by hour
print("\nPrediction Accuracy by Hour of Day:")
hourly_accuracy = df_result.groupby('pickup_hour').agg({
    'absolute_error': 'mean',
    'actual_fare_per_mile': 'mean',
    'predicted_fare_per_mile': 'mean'
}).rename(columns={
    'absolute_error': 'avg_error',
    'actual_fare_per_mile': 'avg_actual_fare',
    'predicted_fare_per_mile': 'avg_predicted_fare'
}).round(4)
print(hourly_accuracy.to_string())

print("\n" + "=" * 60)
print("✓ Analysis complete! Use 'analysis_result' table to create dashboard heatmaps.")
print("=" * 60)